In [1]:
# imports and setup
import pandas as pd

from pathlib import Path
import os


In [2]:
from utils.utility import (
    load_csv, save_parquet, load_parquet,
    normalise, pivot_binary, mineral_columns,
    PROJECT_ROOT, CSV_DIR, PROCESSED_DIR
)


In [9]:
# excel file integration
xlsx_path = "../data/raw/minerals list v2.xlsx"
xls = pd.ExcelFile(xlsx_path)

print(xlsx_path)
for sheet in xls.sheet_names:
    print(sheet)
    df = pd.read_excel(xls, sheet_name=sheet)
    df.to_csv(f"../data/csv/{sheet}.csv", index=False)


../data/raw/minerals list v2.xlsx
00_README
Crit_Min_Frameworks
01_raw_data
02_ref_data
03_global_tot
Mineral Criticality Model
Country Strategic Scores
04_output_report
Mineral Industry List
sector-mineral matrix
sector-country matrix
sector-country list
Country Code
Mineral Codes
Sector Codes
Sources


In [ ]:
# helpers

def normalise(series):
    """Strip + uppercase + string-cast."""
    return series.astype(str).str.strip().str.upper()

def pivot_binary(df, index, columns):
    """Generic binary pivot-table builder."""
    return (
        df.assign(value=1)
          .pivot_table(index=index, columns=columns, values="value", fill_value=0)
    )

def clean_iso(df, col="Country Code"):
    df[col] = df[col].astype(str).str.strip().str.upper()
    return df

In [ ]:
#  mineral-country matrix

df_mc = load_csv("mineral_country_matrix.csv")
df_mc = clean_iso(df_mc, col="Country Code")
df_mc = df_mc.drop(columns=["Total Minerals"])
df_mc = df_mc.set_index("Country Code")


# Identify mineral columns (3‑letter uppercase codes)
mineral_cols = [c for c in df_mc.columns if c.isupper() and len(c) == 3]

# Country totals
country_total_df = (
    df_mc[mineral_cols]
    .sum(axis=1)
    .to_frame("Total Minerals")
    .reset_index()
)

# Mineral totals
mineral_total_df = (
    df_mc[mineral_cols]
    .sum(axis=0)
    .to_frame("Mineral Counts")
    .reset_index()
    .rename(columns={"index": "Mineral Code"})
)
save_parquet(mineral_total_df, "global_mineral_counts.parquet")
save_parquet(country_total_df, "country_mineral_counts.parquet")

# %%
# -----------------------------
# 2. Rebuild mineral-country matrix from mineral_list.csv
# -----------------------------
ml = load_csv("mineral_list.csv")
ml["Countries"] = ml["Countries with Significant Deposits"].str.split(",")
ml["Countries"] = ml["Countries"].apply(lambda lst: [c.strip() for c in lst])
# # ml = clean_iso(ml, col="Country Code")   # optional but safe
# ml["Countries"] = ml["Countries"].apply(lambda lst: [c.strip().upper() for c in lst])
exploded = ml.explode("Countries")


# Country lookup
country_name_to_iso3 = load_csv("country_codes.csv")
country_name_to_iso3 = clean_iso(country_name_to_iso3, col="ISO3")
country_lookup = dict(zip(country_name_to_iso3["Country"], country_name_to_iso3["ISO3"]))

manual_overrides = {
    "DRC": "COD",
    "United States": "USA",
    "Turkiye": "TUR",
    "Morocco (including Western Sahara)": "MAR",
    "North Korea": "PRK",
    "Greenland": "GRL",
}
country_lookup.update(manual_overrides)

exploded["Country Code"] = exploded["Countries"].map(country_lookup)


# Mineral lookup
mineral_name_to_code = load_csv("mineral_codes.csv")
mineral_name_to_code = clean_iso(mineral_name_to_code, col="Code")
mineral_lookup = dict(zip(mineral_name_to_code["Mineral"], mineral_name_to_code["Code"]))
exploded["Mineral"] = exploded["Mineral"].str.replace("*", "", regex=False)
exploded["Mineral Code"] = exploded["Mineral"].map(mineral_lookup)
mineral_country_matrix = pivot_binary(exploded, "Country Code", "Mineral Code")

save_parquet(mineral_country_matrix, "mineral_country_matrix.parquet")

# 3. Mineral–sector matrix
# -----------------------------
sector_codes = load_csv("sector_codes.csv")
sector_codes = clean_iso(sector_codes, col="Sector Code")
all_sectors = normalise(sector_codes['Sector Code'])

mineral_sector = load_csv("mineral_sector_list.csv")
mineral_sector["Sector Code"] = normalise(mineral_sector["Sector Code"])
mineral_sector["Mineral Code"] = normalise(mineral_sector["Mineral Code"])

mat_matrix = pivot_binary(mineral_sector, "Sector Code", "Mineral Code").reindex(all_sectors, fill_value=0)

save_parquet(mat_matrix, "mineral_sector_mat.parquet")

# 4. Sector–country matrix
sector_company = load_csv("sector_company_list.csv")
sector_company = clean_iso(sector_company, col="Country Code")
sector_company = clean_iso(sector_company, col="Sector Code")
sector_company["Sector Code"] = normalise(sector_company["Sector Code"])
sector_company["Country Code"] = normalise(sector_company["Country Code"])
sector_country_matrix = pivot_binary(sector_company, "Sector Code", "Country Code")
sector_country_matrix = sector_country_matrix.reindex(all_sectors, fill_value=0)
save_parquet(sector_country_matrix, "sector_country_matrix.parquet")

# 5. Sector elasticity
sector_meta = load_csv("sector_codes.csv")
sector_elasticity = sector_meta[["Sector Code", "Elasticity"]]
save_parquet(sector_elasticity, "sector_elasticity.parquet")

# 6. Save reference tables

save_parquet(load_csv("sector_company_list.csv"), "sector_company.parquet")
save_parquet(load_csv("sector_codes.csv"), "sector_codes.parquet")
save_parquet(load_csv("mineral_codes.csv"), "mineral_codes.parquet")
save_parquet(load_csv("country_codes.csv"), "country_codes.parquet")


In [ ]:
# geopolitical dataset (Pop, land area, gdp, gni)
# world_population.csv
"""
sources:
https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)
https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population
https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_area
"""

In [ ]:
# Macro dataset
for src, dst in [
    ('world_land.csv', 'world_land_area.parquet'),
    ("world_gni.csv", "world_gni.parquet"),
    ('world_population.csv', 'world_population.parquet'),
    ('world_gdp.csv', 'world_gdp.parquet')
]:
    save_parquet(load_csv(src), dst)

# Load macro datasets
pop = load_parquet("world_population.parquet")
gdp = load_parquet("world_gdp.parquet")
land = load_parquet("world_land_area.parquet")
gni = load_parquet("world_gni.parquet")
country_dim = load_parquet("country_codes.parquet").rename(columns={"ISO3": "Country Code"})

# --- Build macro base ---
macro = (
    country_dim
    .merge(pop, on="Country Code", how="left")
    .merge(gdp, on="Country Code", how="left")
    .merge(land, on="Country Code", how="left")
    .merge(gni, on="Country Code", how="left")
)

In [ ]:
# --- Remove old count columns ---
macro = macro.drop(columns=[
    c for c in macro.columns
    if "Producer Count" in c or "Manufacturer Count" in c
], errors="ignore")

# --- Add mineral producer counts ---
producer_counts = (
    mineral_country_matrix.sum(axis=1)
    .rename("Producer Count")
    .reset_index()
)

# --- Add manufacturer sector counts ---
manufacturer_counts = (
    sector_country_matrix.sum(axis=0)
    .rename("Manufacturer Count")
    .reset_index()
)

# --- Clean merge ---
country_master = (
    macro
    .merge(producer_counts, on="Country Code", how="left")
    .merge(manufacturer_counts, on="Country Code", how="left")
)

save_parquet(country_master, "country_master.parquet")
print("country_master.parquet rebuilt.")

In [ ]:
mineral_country_matrix.head(10)

In [ ]:
"KOR" in sector_country_matrix.columns
sector_country_matrix.loc[:, "KOR"]


In [ ]:
"KOR" in mineral_country_matrix.index


In [ ]:
country_master.head()

In [ ]:
print(country_master)

In [ ]:
# Load the master table
country_master = pd.read_parquet(PROCESSED_DIR / "country_master.parquet")

# Columns to audit
macro_cols = ["Population", "GDP_IMF_2026", "Land_Area_km2", "GNI_Coefficient"]

# Find rows with ANY NaN in these columns
missing_macro = country_master[country_master[macro_cols].isna().any(axis=1)]

# Show results
print("Countries with missing macro data:")
print(missing_macro[["Country", "Country Code"] + macro_cols])
print(f"\nTotal affected countries: {len(missing_macro)}")


In [ ]:
import pandas as pd

# Load all macro datasets
pop = pd.read_parquet(PROCESSED_DIR / "world_population.parquet")
gdp = pd.read_parquet(PROCESSED_DIR / "world_gdp.parquet")
land = pd.read_parquet(PROCESSED_DIR / "world_land_area.parquet")
gni = pd.read_parquet(PROCESSED_DIR / "world_gni.parquet")

country_master = pd.read_parquet(PROCESSED_DIR / "country_master.parquet")

macro_cols = ["Population", "GDP_IMF_2026", "Land_Area_km2", "GNI_Coefficient"]

# Countries missing any macro data
missing = country_master[country_master[macro_cols].isna().any(axis=1)]

audit_rows = []

for _, row in missing.iterrows():
    code = row["Country Code"]
    country = row["Country"]

    audit_rows.append({
        "Country": country,
        "Code": code,
        "Missing Population": code not in pop["Country Code"].values,
        "Missing GDP": code not in gdp["Country Code"].values,
        "Missing Land Area": code not in land["Country Code"].values,
        "Missing GNI": code not in gni["Country Code"].values,
    })

audit_df = pd.DataFrame(audit_rows)

print(audit_df.to_string(index=False))
print(f"\nTotal affected countries: {len(audit_df)}")
